In [73]:
from transformers import AutoProcessor, Blip2ForConditionalGeneration,CLIPTokenizer, CLIPProcessor, CLIPModel
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchinfo import summary
from torchview import  draw_graph
from PIL import Image
from torchviz import make_dot
from urllib.request import urlopen
device = 'cuda'if torch.cuda.is_available() else 'cpu'


In [ ]:
model_id = "openai/clip-vit-base-patch32"

clip_tokenizer = CLIPTokenizer.from_pretrained(model_id)
clip_processor = CLIPProcessor.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id)

clip_model.to(device)

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [50]:

puppy_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/chapter09/images/puppy.png"

image1 = Image.open(urlopen(puppy_path)).convert("RGB")
caption = "a puppy playing in the snow"

In [51]:
# Prepare image tensor
inputs = clip_processor(images=image1, return_tensors="pt").to(device)

# Extract patch embeddings
with torch.no_grad():
    vision_out = clip_model.vision_model(pixel_values=inputs["pixel_values"])
    image_tokens = vision_out.last_hidden_state  # [batch, num_patches+1, hidden_dim]


# Optional: project if you’ll feed them to another model
# proj = torch.nn.Linear(model.config.vision_config.hidden_size ,  out_features=512).to(device)
# image_tokens_proj = proj(image_tokens)

image_tokens.shape

torch.Size([1, 50, 768])

In [52]:
question = "what is shown in the image?"

In [53]:
text_inputs = clip_processor.tokenizer(question, return_tensors="pt").to(device)
text_features = clip_model.text_model(**text_inputs).last_hidden_state
text_features.shape

torch.Size([1, 9, 512])

In [54]:
class CrossAttentionBlock(nn.Module):
    def __init__(self,dim,num_heads = 8):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(dim,num_heads)
        self.ln = nn.LayerNorm(dim)
    def forward(self,text_embeds, image_embeds):
        attn_scores, _ = self.cross_attn(
            query = text_embeds,
            key = image_embeds,
            value = image_embeds
        )
        return self.ln(text_embeds+attn_scores)
    


In [55]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel


tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

decoder_model = GPT2LMHeadModel.from_pretrained("gpt2")

decoder_model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
class SimpleVQAModel(nn.Module):
    def __init__(self, clip_model, decoder_model):
        super().__init__()
        self.clip = clip_model
        self.decoder = decoder_model

        vision_hidden = clip_model.config.vision_config.hidden_size
        text_hidden = decoder_model.config.n_embd

        self.proj = nn.Linear(vision_hidden, text_hidden)
        self.cross_attn = CrossAttentionBlock(text_hidden)

    def forward(self, image, question_inputs, labels=None):
        # Extract features
        with torch.no_grad():
            image_feats = self.clip.vision_model(pixel_values=image).last_hidden_state
            text_feats = self.clip.text_model(**question_inputs).last_hidden_state

        # Project & fuse
        image_feats = self.proj(image_feats)
        fused_feats = self.cross_attn(text_feats, image_feats)

        # Build multimodal prefix for GPT-2
        prefix = fused_feats  # [B, T_img, D]
        question_embeds = self.decoder.transformer.wte(question_inputs["input_ids"])
        inputs_embeds = torch.cat([image_feats,prefix, question_embeds], dim=1)

        # Decode
        outputs = self.decoder(inputs_embeds=inputs_embeds, labels=question_inputs["input_ids"])
        return outputs


In [62]:
vqa_model = SimpleVQAModel(clip_model, decoder_model).to(device)
summary(vqa_model)


Layer (type:depth-idx)                                            Param #
SimpleVQAModel                                                    --
├─CLIPModel: 1-1                                                  1
│    └─CLIPTextTransformer: 2-1                                   --
│    │    └─CLIPTextEmbeddings: 3-1                               25,336,320
│    │    └─CLIPEncoder: 3-2                                      37,828,608
│    │    └─LayerNorm: 3-3                                        1,024
│    └─CLIPVisionTransformer: 2-2                                 --
│    │    └─CLIPVisionEmbeddings: 3-4                             2,398,464
│    │    └─LayerNorm: 3-5                                        1,536
│    │    └─CLIPEncoder: 3-6                                      85,054,464
│    │    └─LayerNorm: 3-7                                        1,536
│    └─Linear: 2-3                                                393,216
│    └─Linear: 2-4                                    

In [72]:
from tqdm import tqdm
# Freeze pretrained components
for p in vqa_model.clip.parameters():
    p.requires_grad = False
for p in vqa_model.decoder.parameters():
    p.requires_grad = False

# Collect trainable parameters
trainable_params = list(vqa_model.proj.parameters()) + list(vqa_model.cross_attn.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# Training loop
def train_vqa(model, dataloader, optimizer, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
        
        for batch in pbar:
            images, questions, answers = batch["image"].to(device), batch["question"], batch["answer"]

            # 1️⃣ CLIP feature extraction (frozen)
            with torch.no_grad():
                image_inputs = clip_processor(images=images, return_tensors="pt").to(device)
                image_feats = model.clip.vision_model(pixel_values=image_inputs["pixel_values"]).last_hidden_state
                question_inputs = clip_processor.tokenizer(questions, return_tensors="pt", padding=True, truncation=True).to(device)
                text_feats = model.clip.text_model(**question_inputs).last_hidden_state

            # 2️⃣ Trainable projection + cross-attention fusion
            image_proj = model.proj(image_feats)
            fused_feats = model.cross_attn(text_feats, image_proj)

            # 3️⃣ Prepare decoder inputs (frozen)
            answer_inputs = tokenizer(answers, return_tensors="pt", padding=True, truncation=True).to(device)
            answer_embeds = model.decoder.transformer.wte(answer_inputs["input_ids"])
            multimodal_embeds = torch.cat([fused_feats, answer_embeds], dim=1)

            # 4️⃣ Decoder forward pass
            outputs = model.decoder(inputs_embeds=multimodal_embeds, labels=answer_inputs["input_ids"])
            loss = outputs.loss

            # 5️⃣ Backprop only through projection + cross-attention
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix({"loss": loss.item()})

        print(f"Epoch {epoch+1} finished | Avg loss = {total_loss / len(dataloader):.4f}")
